# Check Unity Catalog Layer

Lists current Unity Catalog metadata for one selected Ampere layer without changing catalog state.

Steps performed:
1. Select layer and contract path.
2. Load `.env`, UC runtime config, and canonical table contract through `tools/py_utils`.
3. Resolve the schemas assigned to the selected layer.
4. Query Unity Catalog for registered tables in each schema.
5. Print format, column count, and storage location for quick validation.


In [ ]:
# -----------------------------------------------------------------------------
# Manual setup
# -----------------------------------------------------------------------------
LAYER = "bronze"  # bronze | silver | gold
CONTRACT_PATH = "tools/uc/contracts/ampere_tables.json"
VERBOSE = True


In [ ]:
# -----------------------------------------------------------------------------
# Imports and project path setup
# -----------------------------------------------------------------------------
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find project root from current notebook path.")

from tools.py_utils.uc_client import UCClient  # noqa: E402
from tools.py_utils.uc_config import load_uc_bulk_config, uc_client_config  # noqa: E402
from tools.py_utils.uc_contracts import layer_schemas, layers, load_contract  # noqa: E402


In [ ]:
# -----------------------------------------------------------------------------
# Read UC metadata for the selected layer
# -----------------------------------------------------------------------------
contract = load_contract(CONTRACT_PATH)
contract_layers = layers(contract)
if LAYER not in contract_layers:
    raise ValueError(f"Unknown layer {LAYER!r}. Available: {sorted(contract_layers)}")

config = load_uc_bulk_config()
uc = UCClient(uc_client_config(config, verbose=VERBOSE))
registered_tables = []
for schema in layer_schemas(contract, LAYER):
    schema_name = schema["name"]
    tables = uc.list_tables(schema_name)
    print(f"Tables in {uc.config.catalog}.{schema_name}: {len(tables)}")
    for table in tables:
        print(
            f"  - {table['name']:<35} "
            f"{table.get('table_type', '?'):<10} "
            f"{table.get('data_source_format', '?'):<8} "
            f"cols={len(table.get('columns', [])):<3} "
            f"{table.get('storage_location', '')}"
        )
    registered_tables.extend(tables)

registered_tables
